We wrote a script that scans a directory of script text files, detects exact and near duplicates, and organizes them into a structured output. It normalizes text, computes hashes, groups similar files, and produces summary statistics.

In [ ]:
from pathlib import Path
import hashlib, re, unicodedata
from collections import Counter

SOURCE_DIR = Path(r"C:\\Data\\Downloads\\outputforlocal\\dataset\\cleaned_scripts")
DEST_DIR = SOURCE_DIR / "script_deduped_output"

MAX_HAMMING = 6
TOKEN_RATIO_LIMIT = 1.35
TITLE_SIM_THRESHOLD = 0.60

We normalize text to remove formatting differences and tokenize it for analysis.

In [ ]:
def normalize_text(text):
    text = unicodedata.normalize("NFKC", text).lower()
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

def tokenize_text(text):
    return re.findall(r"[a-z0-9']+", text)

We compute SHA256 for exact matches and simhash for approximate similarity.

In [ ]:
def sha256(text):
    return hashlib.sha256(text.encode()).hexdigest()

def simhash(tokens):
    v = [0]*64
    for token, w in Counter(tokens).items():
        h = int.from_bytes(hashlib.blake2b(token.encode(), digest_size=8).digest(), "big")
        for i in range(64):
            v[i] += w if (h>>i)&1 else -w
    return sum((1<<i) for i in range(64) if v[i]>0)

def hamming(a,b):
    return (a^b).bit_count()

We define similarity based on simhash distance and token ratio.

In [ ]:
def content_near(d1, d2):
    dist = hamming(d1["sim"], d2["sim"])
    ratio = max(d1["tokens"], d2["tokens"]) / max(1, min(d1["tokens"], d2["tokens"]))
    return dist <= MAX_HAMMING and ratio <= TOKEN_RATIO_LIMIT

We process files into records with hashes and statistics.

In [ ]:
data = []
for path in SOURCE_DIR.rglob("*.txt"):
    try:
        raw = path.read_text(errors="ignore")
        norm = normalize_text(raw)
        tokens = tokenize_text(norm)
        data.append({
            "path": path,
            "hash": sha256(norm),
            "sim": simhash(tokens),
            "tokens": len(tokens)
        })
    except:
        pass

len(data)

These clusters can still mix different scripts, so we refine them using filename similarity based on overlap of title tokens.

Each final group is labeled as unique, exact duplicate, or near duplicate.

We then copy files into output folders. Unique files go into one folder. Duplicate groups get numbered folders. For each duplicate group, we select the largest file as canonical and also place a copy of it in the unique folder as a representative.

Finally, we write JSON files that record all group assignments, summary counts, and any read failures.

Final computed statistics from the run:

Total files: 8,005


Unique files: 4,225 (52.78%) 

Exact duplicates: 306 (3.82%) 

Near duplicates: 3,474 (43.40%)


Exact groups: 143 

Near groups: 600